# 13 · The paper's daily interval-decomposition features in our pipeline (E13)

Tian, Yang, Wei & Wang (2026) build weekly [min, max] intervals of BTC, gold, WTI, DXY, S&P 500, Google Trends and news sentiment, decompose them jointly with NA-MEMD, group the IMFs into low / medium / high frequency by sample entropy and model each band separately. E13 builds the same features **daily and causally** (the decomposition is recomputed every day on past data only) with `rltrade.paper_daily`, lags them one UTC day onto our 1-minute bars and gives them to the E5 agent (DDQN, E6 hyperparameters, E4's GRU factors).

States: **Gv / Gd** = A + the 12 features with VADER / DeepSeek sentiment (dim 22); **Hv / Hd** = G + gate (no opening or enlarging while yesterday's sentiment interval is wider than its train median). A and D are E5's runs.

**Input (upload once, never committed):** `thesis_rl/prepared_e13/` = `market_{train,val,test}.npz` + `meta.json` from `python -m rltrade.paper_daily prepare`. The runs resume: re-run the run cell after a disconnect, finished runs are skipped. `GROUPS` / `SEEDS` can be split across sessions.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/12-paper-intervals"
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
seeding.record_environment(L.report / "environment", "13_paper_daily")

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from rltrade.env import EnvConfig
from rltrade.runner import RunSpec, run_seeds
from rltrade import results as R
PREP = L.root / "prepared_e13"
assert (PREP / "market_train.npz").exists(), f"upload prepared_e13 to {PREP}"
print(json.dumps(json.loads((PREP / "meta.json").read_text())["e13"], indent=1))
SEEDS = [0, 1, 2, 3, 4]
GROUPS = ("Gd", "Hd")                     # DeepSeek first (the E5 scorer); then set ("Gv", "Hv") for VADER
TOTAL_STEPS, EVAL_EVERY, MIN_TRADES = 2_000_000, 100_000, 5
AGENT, EXP = "ddqn", "13_paper_daily"
hp = json.loads((L.runs / "tuning" / f"best_hp_mt{MIN_TRADES}.json").read_text())[AGENT]
AGENT_HP, ENV_OVER = hp["agent_hp"], hp["env"]
F = paths.factor_dir(L)                   # E4-selected GRU (gru_s1), as in E5
print("agent:", AGENT, AGENT_HP, ENV_OVER)

In [ ]:
for state in GROUPS:
    run_seeds(RunSpec(EXP, AGENT, env=EnvConfig(state=state, **ENV_OVER), agent_hp=AGENT_HP, min_val_trades=MIN_TRADES,
                      total_steps=TOTAL_STEPS, eval_every=EVAL_EVERY), SEEDS, PREP, F, L.runs)

## Results against E5 (test, net of costs)

In [ ]:
res = R.load_results(L.runs)
e13 = res[(res.experiment == EXP) & (res.agent == AGENT)]
a = res[(res.experiment == f"05_ablation_{AGENT}_mt{MIN_TRADES}") & (res.agent == AGENT) & (res.state == "A") & (res.signal_source == "lagged")]
d = res[(res.experiment == f"06_agents_mt{MIN_TRADES}") & (res.agent == AGENT)]
ab = pd.concat([a, d, e13], ignore_index=True)
sm = R.summarise(ab)
table = R.markdown_table(sm, group=("state",))
print(table)
def compare(x, y, metric="test_sharpe"):
    u, v = ab[ab.state == x][metric].values, ab[ab.state == y][metric].values
    t = stats.ttest_ind(u, v, equal_var=False)
    return {"pair": f"{x} - {y}", "diff_mean": u.mean() - v.mean(), "welch_t": t.statistic, "p": t.pvalue, "n": (len(u), len(v))}
pairs = [(g, "A") for g in GROUPS] + [(g, "D") for g in GROUPS] + [("Hd", "Gd"), ("Hv", "Gv"), ("Gd", "Gv")]
tests = pd.DataFrame([compare(*p) for p in pairs if (ab.state == p[0]).any() and (ab.state == p[1]).any()])
print(tests.round(4))       # n = 5 per group: indicative only
(L.report / "e13_paper_daily.md").write_text("# E13, paper daily features (test, net of costs)

" + table + "

"
                                              + R.df_markdown(tests, 4) + "
", encoding="utf-8")